Estimate the influence of spatial factors on the expenditure/income distribution for the admin units at the AOI.

### Authorship
@author: Alexandre Pereira Santos <br>
alexandre.santos(at)lmu.de<br>
- uses pandas, geopandas, rasterio, rasterstats
- may include content from ChatGPT or GitHub Copilot

### Features
1. Load admin units for the AOi (e.g., regencies and cities in JAK)
2. Load income or expenditure distribution 
3. Calculate population under each income bracket
4. Load spatial covariates:
   1. Population density
   2. Built up 
   3. Distance to roads
   4. Distance to schools
   5. Distance to suburban centres
   6. Distance to shopping malls
   7. Distance to markets
   8. Distance to airports
   9. Distance to CBD
   10. Distance to attractive residential areas
   11. Distance to water bodies
   12. Slope
   13. Hazards
5.  Quantify the covariates for each admin unit (i.e., count, add, or average) using areal stats
6.  Calculate the regression for each admin unit:
    1.  Dependent variable: number of people per expenditure quintile (reference from the metropolitan scale) for each GADM level 2 unit (city or regency)
    2.  Independent variables: spatial covariates (above)

# Imports

In [2]:
import pandas as pd
import geopandas as gpd

from rasterstats import zonal_stats

import os
from pathlib import Path
import fnmatch
import shutil

import numpy as np
import random 
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

from shapely.geometry.polygon import Polygon
from shapely.geometry import MultiLineString, LineString
import json

import rasterio
from rasterio.mask import mask
from rasterio.plot import show, show_hist, plotting_extent
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.features import dataset_features

#from libpysal.weights import lat2W, Rook
#from esda.moran import Moran

# init

In [3]:
#input a vector and a raster file for each city
AOI_path = Path('../data/processed/')
MUM_file = 'MUM_LIM_reference_AOI_30m.shp' # APS: updated AOI 01.08.2024
MAN_file = 'MAN_LIM_reference_AOI_30m.shp' # APS: updated AOI 01.11.2024
JAK_file = 'JAK_LIM_reference_AOI_30m.shp' # APS: updated AOI 23.08.2024
#  the 'case_city' variable defines the city we are working on. APS: Need to update the code below to change it from MUM to the variable name

case_city = 'JAK' #MUM MAN JAK

if case_city == 'MUM':
    
    AOI_gdf = gpd.read_file(AOI_path / MUM_file) #.to_crs(epsg=4326)
    AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

    ref_raster_path = Path('../data/processed/MUM_LIM_reference_AOI_30m.tif')
    ref_raster_150_path = Path('../data/processed/MUM_LIM_reference_AOI_150m.tif')
    

    raw_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\raw\\')
    interim_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\interim\\')
    processed_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\processed\\')
    external_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\external\\')
    model_inputs_30m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\model_inputs\\TIFF_30m\\')
    model_inputs_150m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\model_inputs\\TIFF_150m\\')
    model_inputs_SLEUTH_150m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\model_inputs\\TIFF_SLEUTH_150m')
    ascii150_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\model_inputs\\ASCII_150m')
    ascii150_SLEUTH_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MUM\\model_inputs\\ASCII_SLEUTH_150m')
    ti_city_ascii_path = Path('../model/MUM/data/in/')
    ti_city_out_path = Path('../model/MUM/data/out/')
    ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

    gadm_var = 'NAME_3'

if case_city == 'MAN':
    
    AOI_gdf = gpd.read_file(AOI_path / MAN_file) #.to_crs(epsg=4326)
    AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

    ref_raster_path = Path('../data/processed/MAN_LIM_reference_AOI_30m.tif')
    ref_raster_150_path = Path('../data/processed/MAN_LIM_reference_AOI_150m.tif')

    raw_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\raw\\')
    interim_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\interim\\')
    processed_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\processed\\')
    external_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\external\\')
    model_inputs_30m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\model_inputs\\TIFF_30m\\')
    model_inputs_150m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\model_inputs\\TIFF_150m\\')
    model_inputs_SLEUTH_150m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\model_inputs\\TIFF_SLEUTH_150m')
    ascii150_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\model_inputs\\ASCII_150m')
    ascii150_SLEUTH_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\MAN\\model_inputs\\ASCII_SLEUTH_150m')
    ti_city_ascii_path = Path('../model/MAN/data/in/')
    ti_city_out_path = Path('../model/MAN/data/out/')
    ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

    gadm_var = 'NAME_2'

if case_city == 'JAK':
    
    AOI_gdf = gpd.read_file(AOI_path / JAK_file) #.to_crs(epsg=4326)
    AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

    ref_raster_path = Path('../data/processed/JAK_LIM_reference_AOI_30m.tif')
    ref_raster_150_path = Path('../data/processed/JAK_LIM_reference_AOI_150m.tif')

    raw_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\raw\\')
    interim_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\interim\\')
    processed_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\processed\\')
    external_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\external\\')
    model_inputs_30m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\model_inputs\\TIFF_30m\\')
    model_inputs_150m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\model_inputs\\TIFF_150m\\')
    model_inputs_SLEUTH_150m_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\model_inputs\\TIFF_SLEUTH_150m')
    ascii150_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\model_inputs\\ASCII_150m')
    ascii150_SLEUTH_path = Path(r'D:\\Dropbox\\x\\PostDoc\\23 10 LMU\\13 exp_cases\\24 01 Urban SSP MAN MUM JAK\\05_GIS_temp\\JAK\\model_inputs\\ASCII_SLEUTH_150m')
    ti_city_ascii_path = Path('../model/JAK/data/in/')
    ti_city_out_path = Path('../model/JAK/data/out/')
    ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

    gadm_var = 'NAME_2'

external_data_path = Path(r'D:\\Dropbox\\x\\_data')
ascii150_arcgis_path = Path(r"D:\\Dropbox\\x\\PostDoc\ASCII_150m")

%run ./ti_city_00_raster_functions.ipynb

# read the reference raster
with rasterio.open(ref_raster_path,'r') as src:
    ref_raster = src
    ref_meta = src.meta
    ref_height, ref_width, ref_area = get_transform(ref_raster)



# suppress deprecation warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

# 1. Load admin units for the AOi

In [4]:
# import the GADM4 level 3 polygons from the shapefile
gadm_file_name = case_city + '_LIM_admin_boundaries_GADM_A.shp'

gadm_path = external_path / gadm_file_name
gadm_gdf_temp = gpd.read_file(gadm_path)

#print(gadm_gdf.columns)

# drop the columns with "auxiliar" in their name
gadm_gdf = gadm_gdf_temp.drop(columns=[col for col in gadm_gdf_temp.columns if "auxiliar" in col])

gadm_gdf.head()



,OBJECTID,Shape_Leng,Shape_Area,AREA_NAME,GEO_CONCAT,GEO_MATCH,CNTRY_NAME,ADM1_NAME,ADM2_NAME,ADM_LEVEL,USCBCMNT,NSO_CODE,ADM2_NmEn,area_km2,geometry
0,157.0,0.838971,0.011834,KOTA JAKARTA SELATAN,INDONESIA_JAKARTA_KOTA JAKARTA SELATAN,IDN_06_02,INDONESIA,JAKARTA,KOTA JAKARTA SELATAN,2,None,3171,South Jakarta,144.813,"POLYGON ((701679.875 9314041.041, 701730.632 9..."
1,158.0,1.021113,0.015089,KOTA JAKARTA TIMUR,INDONESIA_JAKARTA_KOTA JAKARTA TIMUR,IDN_06_03,INDONESIA,JAKARTA,KOTA JAKARTA TIMUR,2,None,3172,East Jakarta,184.658,"POLYGON ((714679.786 9319532.713, 714685.73 93..."
2,159.0,0.383759,0.003913,KOTA JAKARTA PUSAT,INDONESIA_JAKARTA_KOTA JAKARTA PUSAT,IDN_06_04,INDONESIA,JAKARTA,KOTA JAKARTA PUSAT,2,None,3173,Central Jakarta,47.887,"POLYGON ((702582.551 9321484.718, 702582.94 93..."
3,160.0,0.590340,0.010251,KOTA JAKARTA BARAT,INDONESIA_JAKARTA_KOTA JAKARTA BARAT,IDN_06_05,INDONESIA,JAKARTA,KOTA JAKARTA BARAT,2,None,3174,West Jakarta,125.470,"POLYGON ((688670.713 9325974.502, 688673.262 9..."
4,161.0,1.143005,0.011459,KOTA JAKARTA UTARA,INDONESIA_JAKARTA_KOTA JAKARTA UTARA,IDN_06_06,INDONESIA,JAKARTA,KOTA JAKARTA UTARA,2,None,3175,North Jakarta,140.264,"MULTIPOLYGON (((707699.026 9325346.397, 707609..."


# 2. Load & calculate population based on income or expenditure distribution

In [8]:
population_file = case_city + '_POP_population change.xlsx'

# get the model_inputs_150m_path parent folder
population_path = model_inputs_150m_path.parent / population_file
population_df = pd.read_excel(population_path, sheet_name='raw')
population_df.head()

#calculate the per capita income for every quintile using the 2020 population
population_df['pc_income_Q1_2020'] = (population_df['inc_20QTL']*1000) / (population_df['Population_2020']*0.2)
population_df['pc_income_Q2_2020'] = (population_df['inc_40QTL']*1000) / (population_df['Population_2020']*0.2)
population_df['pc_income_Q3_2020'] = (population_df['inc_60QTL']*1000) / (population_df['Population_2020']*0.2)
population_df['pc_income_Q4_2020'] = (population_df['inc_80QTL']*1000) / (population_df['Population_2020']*0.2)
population_df['pc_income_Q5_2020'] = (population_df['inc_100QTL']*1000) / (population_df['Population_2020']*0.2)
population_df[['pc_income_Q1_2020',
                     'pc_income_Q2_2020',
                     'pc_income_Q3_2020',
                     'pc_income_Q4_2020',
                     'pc_income_Q5_2020']].head()

,pc_income_Q1_2020,pc_income_Q2_2020,pc_income_Q3_2020,pc_income_Q4_2020,pc_income_Q5_2020
0,182.853963,422.224133,886.734696,1430.877945,2276.333196
1,211.787771,628.468626,1022.223058,1207.880709,1971.803386
2,52.543272,343.823669,1159.289152,1517.292286,1972.145585
3,61.994195,411.948627,1037.362953,1259.889338,2311.005901
4,151.980961,339.027329,893.803647,1563.437948,2240.128098


In [11]:
# calculate the min, max, median per capita income for the whole dataset
pc_income_cols = ['pc_income_Q1_2020', 'pc_income_Q2_2020', 'pc_income_Q3_2020', 
                  'pc_income_Q4_2020', 'pc_income_Q5_2020']

min_income = population_df[pc_income_cols].min().min()
max_income = population_df[pc_income_cols].max().max()
median_income = population_df[pc_income_cols].median().median()
standard_deviation_income = population_df[pc_income_cols].std().std()

print(f"\nPer Capita Income Statistics:")
print(f"Minimum: ${min_income:,.2f}")
print(f"Maximum: ${max_income:,.2f}")
print(f"Median: ${median_income:,.2f}")
print(f"Standard Deviation: ${standard_deviation_income:,.2f}")

# set cut-offs for income classes based on standard deviation from the median
deprived_income_cutoff = median_income - standard_deviation_income 
low_income_cutoff = median_income - (standard_deviation_income / 2)
middle_income_cutoff = median_income + (standard_deviation_income / 2)
high_income_cutoff = median_income + standard_deviation_income

cuttoff_list = [deprived_income_cutoff, low_income_cutoff, middle_income_cutoff, high_income_cutoff]
for i in cuttoff_list:
    print(f"Cut-off: ${i:,.2f}")



Per Capita Income Statistics:
Minimum: $52.54
Maximum: $2,636.29
Median: $1,022.22
Standard Deviation: $190.23
Cut-off: $831.99
Cut-off: $927.11
Cut-off: $1,117.34
Cut-off: $1,212.46


In [ ]:
import numpy as np
from scipy.stats import lognorm
from scipy.optimize import minimize

# Inputs (your data)
obs = np.array([81598.53089, 188417.4031, 395705.114, 638528.8894, 1015813.062])  # total expenditure per quintile
N = 2231249

q_pop = N * 0.2  # population per quintile

# Cutoffs (value ranges are: (-inf, c1], (c1,c2], (c2,c3], (c3,c4], (c4, +inf))
cutoffs = [831.99, 927.11, 1117.34, 1212.46]

def model_totals(params):
    s, mu = params
    q_edges = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    tot = []
    for a, b in zip(q_edges[:-1], q_edges[1:]):
        va = lognorm.ppf(max(a, 1e-12), s, scale=np.exp(mu))
        vb = lognorm.ppf(min(b, 1-1e-12), s, scale=np.exp(mu))
        # expected per-person expenditure in [va, vb]
        mean_seg = lognorm.expect(lambda x: x, args=(s,), scale=np.exp(mu), lb=va, ub=vb)
        tot.append(N * mean_seg * 1.0)  # total expenditure in that population quintile
    return np.array(tot)

def loss(params):
    # penalize invalid params
    s, mu = params
    if s <= 0:
        return 1e30
    return np.sum((model_totals(params) - obs) ** 2)

# initial guess: s~1, mu = log(mean per person)
mean_per_person = obs.sum() / N
init = [1.0, np.log(max(mean_per_person, 1e-6))]
res = minimize(loss, x0=init, method='Nelder-Mead')

if not res.success:
    raise RuntimeError("Optimization failed: " + res.message)

s_fit, mu_fit = res.x

# compute fractions in target value ranges
edges = [-np.inf] + cutoffs + [np.inf]
fracs = []
for lo, hi in zip(edges[:-1], edges[1:]):
    lo_val = lo if lo != -np.inf else -1e9
    hi_val = hi if hi != np.inf else 1e9
    frac = lognorm.cdf(hi_val, s_fit, scale=np.exp(mu_fit)) - lognorm.cdf(lo_val, s_fit, scale=np.exp(mu_fit))
    fracs.append(frac)

counts = np.array(fracs) * N

# Output
for i, (lo, hi, c) in enumerate(zip(edges[:-1], edges[1:], counts), 1):
    label = f"Class {i}: ({lo if lo!=-np.inf else '-inf'} , {hi if hi!=np.inf else '+inf'}]"
    print(f"{label:40} -> estimated count = {int(round(c))}")

# print fitted params for diagnostics
print("\nFitted lognormal params: s =", s_fit, "mu =", mu_fit)
print("Implied per-person mean:", lognorm.mean(s_fit, scale=np.exp(mu_fit)))


Class 1: (-inf , 831.99]                 -> estimated count = 2231249
Class 2: (831.99 , 927.11]               -> estimated count = 0
Class 3: (927.11 , 1117.34]              -> estimated count = 0
Class 4: (1117.34 , 1212.46]             -> estimated count = 0
Class 5: (1212.46 , +inf]                -> estimated count = 0

Fitted lognormal params: s = 0.6942870999123228 mu = -0.18059502525359616
Implied per-person mean: 1.0622850158276176


In [ ]:
# Python: fit lognormal to quintile total expenditures and get counts for arbitrary value cutoffs
import numpy as np
from scipy.stats import lognorm
from scipy.optimize import minimize

# input (example values you gave)
obs = np.array([81598.53089, 188417.4031, 395705.114, 638528.8894, 1015813.062])  # totals per quintile
population = population_df['Population_2020'].sum()  # total population

# model: total expenditure in each 20%-population quintile for (s, mu)
def model_totals(params):
    s, mu = params
    q_edges = [0,0.2,0.4,0.6,0.8,1.0]
    tot = []
    for a,b in zip(q_edges[:-1], q_edges[1:]):
        va = lognorm.ppf(a, s, scale=np.exp(mu))
        vb = lognorm.ppf(b, s, scale=np.exp(mu))
        # expected per-person expenditure in [va,vb]
        mean_seg = lognorm.expect(lambda x: x, args=(s,), scale=np.exp(mu), lb=va, ub=vb)
        tot.append(population * mean_seg)
    return np.array(tot)

def loss(params):
    return np.sum((model_totals(params) - obs)**2)

# initial guess
res = minimize(loss, x0=[1.0, np.log(obs.sum()/population)])
s_fit, mu_fit = res.x

# get counts for any expenditure value range [x_lo, x_hi]
def counts_in_range(x_lo, x_hi):
    frac = lognorm.cdf(x_hi, s_fit, scale=np.exp(mu_fit)) - lognorm.cdf(x_lo, s_fit, scale=np.exp(mu_fit))
    return population * frac


# 3. Load spatial covariates:
   1. Population density
   2. Built up 
   3. Distance to roads
   4. Distance to schools
   5. Distance to suburban centres
   6. Distance to shopping malls
   7. Distance to markets
   8. Distance to airports
   9. Distance to CBD
   10. Distance to attractive residential areas
   11. Distance to water bodies
   12. Slope
   13. Hazards

## Imports

## Quantify the covariates for each admin unit

# 4. Regression

## Compile descriptive stats

# 5. Export

# The end (for now).